In [2]:
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
demand = pd.read_csv("../data/processed/demand_cleaned.csv")

In [4]:
demand["Date"] = pd.to_datetime(demand["Date"])

In [5]:
demand["Year"] = demand["Date"].dt.year
demand["Month"] = demand["Date"].dt.month
demand["Day"] = demand["Date"].dt.day
demand["DayOfWeek"] = demand["Date"].dt.dayofweek

In [6]:
demand.head()

,Product ID,Date,Store ID,Sales Quantity,Price,Promotions,Seasonality Factors,External Factors,Demand Trend,Customer Segments,Year,Month,Day,DayOfWeek
0,4277,2024-01-03,48,330,24.38,No,Festival,Competitor Pricing,Increasing,Regular,2024,1,3,2
1,5540,2024-04-29,10,334,74.98,Yes,Holiday,Weather,Stable,Premium,2024,4,29,0
2,5406,2024-01-11,67,429,24.83,Yes,Holiday,Economic Indicator,Decreasing,Premium,2024,1,11,3
3,5617,2024-04-04,17,298,13.41,No,NaN,Economic Indicator,Stable,Regular,2024,4,4,3
4,3480,2024-12-14,33,344,94.96,Yes,Festival,Weather,Increasing,Regular,2024,12,14,5


Encode Categorical Variables

In [7]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = [
    "Promotions",
    "Seasonality Factors",
    "External Factors",
    "Demand Trend",
    "Customer Segments"
]

encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    demand[col] = le.fit_transform(demand[col])
    encoders[col] = le

Select Features and Target

Target

In [8]:
y = demand["Sales Quantity"]

Features

In [9]:
X = demand.drop(
    columns=[
        "Sales Quantity",
        "Date"
    ]
)

In [10]:
print(X.shape)
print(y.shape)

(10000, 12)
(10000,)


Train test Split

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
print(X_train.shape)
print(X_test.shape)

(8000, 12)
(2000, 12)


Model (Linear Regression)

In [13]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

In [14]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MAE:", mean_absolute_error(y_test, pred_lr))
print("RMSE:", mean_squared_error(y_test, pred_lr) ** 0.5)
print("R2:", r2_score(y_test, pred_lr))

MAE: 125.65164813358065
RMSE: 145.25406014867852
R2: -0.002366003782010795


In [15]:
correlation = demand.corr(numeric_only=True)

correlation["Sales Quantity"].sort_values(ascending=False)

Sales Quantity         1.000000
Store ID               0.015043
Seasonality Factors    0.013690
Customer Segments      0.013230
Promotions             0.011815
Demand Trend           0.005904
DayOfWeek              0.002045
Product ID             0.001199
Price                 -0.006155
External Factors      -0.008442
Month                 -0.008683
Day                   -0.014251
Year                        NaN
Name: Sales Quantity, dtype: float64

Tree Model

In [16]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

print("MAE:", mean_absolute_error(y_test, pred_rf))
print("RMSE:", mean_squared_error(y_test, pred_rf) ** 0.5)
print("R2:", r2_score(y_test, pred_rf))

MAE: 126.43412000000001
RMSE: 146.61034241937028
R2: -0.021172200789302265
